# 03 — Graph EDA and Feature Engineering

This notebook explores transaction behavior in the PaySim graph and engineers
pre-transaction features for fraud detection.

The analysis combines:

- transaction-level behavior
- graph structure
- historical receiver activity
- temporal graph activity

Only information available before the current transaction is used in the final
modeling feature set.

## 1. Load Canonical Transactions

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

from neo4j import GraphDatabase
from getpass import getpass

DATA_DIR = Path("../data")

df = pd.read_csv(
    DATA_DIR / "paysim_canonical_200k.csv"
)

print("Shape:", df.shape)
print("Missing values:", df.isna().sum().sum())

df.head()

Shape: (200000, 12)
Missing values: 0


,tx_id,time_step,tx_type,amount,src_id,src_balance_before,src_balance_after,dst_id,dst_balance_before,dst_balance_after,label_fraud,label_flagged
0,tx_1,207,CASH_OUT,201544.43,C1446786192,0.0,0.0,C629162506,4427398.04,4628942.46,0,0
1,tx_2,301,CASH_OUT,99796.32,C1507137317,0.0,0.0,C95898629,2244500.63,2344296.95,0,0
2,tx_3,429,CASH_OUT,31901.99,C689120377,0.0,0.0,C1777573992,9080459.95,9112361.94,0,0
3,tx_4,187,CASH_OUT,286477.31,C1224308913,0.0,0.0,C766410750,949477.95,1235955.26,0,0
4,tx_5,159,CASH_OUT,119075.98,C2098350559,0.0,0.0,C85657936,187137.64,306213.62,0,0


## 2. Connect to the Transaction Graph

In [4]:
from neo4j import GraphDatabase
from getpass import getpass

URI = "bolt://127.0.0.1:7687"
USER = "neo4j"
DATABASE = "neo4j"

PASSWORD = getpass("Neo4j password: ")

driver = GraphDatabase.driver(
    URI,
    auth=(USER, PASSWORD)
)

driver.verify_connectivity()

print("Neo4j connection successful.")

Neo4j password:  ········


Neo4j connection successful.


## 3. Structural Graph EDA

### 3.1 Sender Activity

We first inspect whether fraudulent transactions are concentrated among
repeated sender accounts.

In [5]:
with driver.session(database=DATABASE) as session:
    result = session.run("""
    MATCH (src:Account)-[:SENT]->(t:Transaction)
    WHERE t.label_fraud = 1

    RETURN
        count(t) AS fraud_transactions,
        count(DISTINCT src) AS unique_fraud_senders
    """)

    sender_fraud_summary = dict(result.single())

sender_fraud_summary

{'fraud_transactions': 8213, 'unique_fraud_senders': 8213}

In [6]:
with driver.session(database=DATABASE) as session:
    result = session.run("""
    MATCH (src:Account)-[:SENT]->(t:Transaction)

    WITH src, count(t) AS transaction_count

    RETURN
        transaction_count,
        count(*) AS account_count

    ORDER BY transaction_count
    """)

    sender_activity_df = pd.DataFrame(
        [dict(record) for record in result]
    )

sender_activity_df

,transaction_count,account_count
0,1,199982
1,2,9


**Observation:** Sender history is extremely sparse in the sampled graph.
Almost every sender appears only once. Sender-degree features are therefore
unlikely to provide useful behavioral history in this dataset.

This sparsity is partly sample-dependent and should not be assumed to hold
for other transaction datasets.

### 3.2 Receiver Activity

Receiver accounts show slightly more repeated interaction than sender accounts,
so receiver-side graph history is investigated further.

In [7]:
with driver.session(database=DATABASE) as session:
    result = session.run("""
    MATCH (t:Transaction)-[:TO]->(dst:Account)
    WHERE t.label_fraud = 1

    WITH dst, count(t) AS fraud_received_count

    RETURN
        count(dst) AS unique_fraud_receivers,
        max(fraud_received_count) AS max_fraud_received,
        avg(fraud_received_count) AS avg_fraud_received
    """)

    receiver_fraud_summary = dict(result.single())

receiver_fraud_summary

{'unique_fraud_receivers': 8169,
 'max_fraud_received': 2,
 'avg_fraud_received': 1.005386216183128}

In [8]:
with driver.session(database=DATABASE) as session:
    result = session.run("""
    MATCH (t:Transaction)-[:TO]->(dst:Account)
    WHERE t.label_fraud = 1

    WITH dst, count(t) AS fraud_received_count

    RETURN
        fraud_received_count,
        count(*) AS receiver_count

    ORDER BY fraud_received_count
    """)

    fraud_receiver_distribution = pd.DataFrame(
        [dict(record) for record in result]
    )

fraud_receiver_distribution

,fraud_received_count,receiver_count
0,1,8125
1,2,44


**Observation:** Receiver-side behavior contains more historical structure than
sender-side behavior. This motivates the receiver-history graph features used
later in the notebook.

## 4. Transaction Amount and Type Analysis

In [10]:
amount_summary = (
    df.groupby(["label_fraud", "tx_type"])["amount"]
    .agg(
        count="count",
        mean="mean",
        median="median",
        p95=lambda x: x.quantile(0.95),
        max="max"
    )
    .round(2)
)

amount_summary

count        mean     median         p95          max
label_fraud tx_type                                                         
0           CASH_OUT  155046   174099.26  147444.64   425939.69   2393198.44
            TRANSFER   36741   905879.51  484786.08  2639316.01  92445516.64
1           CASH_OUT    4116  1455102.59  435516.90  7841465.04  10000000.00
            TRANSFER    4097  1480891.67  445705.76  8098658.15  10000000.00

## 5. Balance Consistency Analysis

Balance changes are inspected to identify potentially useful behavioral signals
and dataset-specific artifacts.

Post-transaction balances are analyzed diagnostically, but they are excluded
from the primary pre-transaction modeling feature set.

In [12]:
balance_analysis = df.copy()

balance_analysis["sender_balance_error"] = (
    (
        balance_analysis["src_balance_before"]
        - balance_analysis["amount"]
        - balance_analysis["src_balance_after"]
    )
    .abs()
)

balance_analysis["receiver_balance_error"] = (
    (
        balance_analysis["dst_balance_before"]
        + balance_analysis["amount"]
        - balance_analysis["dst_balance_after"]
    )
    .abs()
)

balance_summary = (
    balance_analysis
    .groupby(["label_fraud", "tx_type"])
    [["sender_balance_error", "receiver_balance_error"]]
    .agg(["mean", "median"])
    .round(2)
)

balance_summary

sender_balance_error            receiver_balance_error  \
                                     mean     median                   mean   
label_fraud tx_type                                                           
0           CASH_OUT            147943.89  119990.55               29728.16   
            TRANSFER            872413.97  450485.58              104092.55   
1           CASH_OUT              1306.13       0.00               13965.04   
            TRANSFER             20122.05       0.00             1479702.98   

                                 
                         median  
label_fraud tx_type              
0           CASH_OUT       0.00  
            TRANSFER       0.00  
1           CASH_OUT       0.00  
            TRANSFER  443893.69

In [13]:
zero_error_summary = (
    balance_analysis
    .assign(
        sender_error_zero=(
            balance_analysis["sender_balance_error"] == 0
        ),
        receiver_error_zero=(
            balance_analysis["receiver_balance_error"] == 0
        )
    )
    .groupby(["label_fraud", "tx_type"])
    [["sender_error_zero", "receiver_error_zero"]]
    .mean()
    .mul(100)
    .round(2)
)

zero_error_summary

sender_error_zero  receiver_error_zero
label_fraud tx_type                                         
0           CASH_OUT               7.01                58.18
            TRANSFER               2.19                57.30
1           CASH_OUT              99.42                70.19
            TRANSFER              97.49                 0.00

### Leakage / artifact decision

Post-transaction balances can reveal information that may not be available when
a fraud decision must be made.

In addition, PaySim shows unusually strong balance-consistency patterns for
fraudulent transactions. These may reflect synthetic-data generation rules
rather than general fraud behavior.

Therefore:

- `src_balance_after`
- `dst_balance_after`
- `sender_balance_error`
- `receiver_balance_error`

are excluded from the primary pre-transaction model.

## 6. Transaction-Level Feature Engineering

Pre-transaction features are derived from transaction amount and the sender's
available balance.

These features are designed to be available at decision time and avoid using
post-transaction information.

In [14]:
transaction_features = df[[
    "tx_id",
    "src_id",
    "dst_id",
    "time_step",
    "tx_type",
    "amount",
    "src_balance_before",
    "dst_balance_before",
    "label_fraud"
]].copy()

print("Base feature table:", transaction_features.shape)

transaction_features.head()

Base feature table: (200000, 9)


,tx_id,src_id,dst_id,time_step,tx_type,amount,src_balance_before,dst_balance_before,label_fraud
0,tx_1,C1446786192,C629162506,207,CASH_OUT,201544.43,0.0,4427398.04,0
1,tx_2,C1507137317,C95898629,301,CASH_OUT,99796.32,0.0,2244500.63,0
2,tx_3,C689120377,C1777573992,429,CASH_OUT,31901.99,0.0,9080459.95,0
3,tx_4,C1224308913,C766410750,187,CASH_OUT,286477.31,0.0,949477.95,0
4,tx_5,C2098350559,C85657936,159,CASH_OUT,119075.98,0.0,187137.64,0


In [15]:
transaction_features["log_amount"] = np.log1p(
    transaction_features["amount"]
)

In [16]:
transaction_features["amount_to_src_balance_ratio"] = np.where(
    transaction_features["src_balance_before"] > 0,
    transaction_features["amount"]
    / transaction_features["src_balance_before"],
    0
)

In [17]:
transaction_features["amount_exceeds_src_balance"] = (
    transaction_features["amount"]
    > transaction_features["src_balance_before"]
).astype(int)

In [18]:
transaction_features["src_balance_zero"] = (
    transaction_features["src_balance_before"] == 0
).astype(int)

In [19]:
transaction_features_v1 = transaction_features[[
    "tx_id",
    "src_id",
    "dst_id",
    "time_step",
    "tx_type",

    "amount",
    "log_amount",
    "src_balance_before",
    "dst_balance_before",
    "amount_to_src_balance_ratio",
    "amount_exceeds_src_balance",
    "src_balance_zero",

    "label_fraud"
]].copy()

print("Shape:", transaction_features_v1.shape)
print("Missing:", transaction_features_v1.isna().sum().sum())
print(
    "Duplicate tx_id:",
    transaction_features_v1["tx_id"].duplicated().sum()
)

transaction_features_v1.head()

Shape: (200000, 13)
Missing: 0
Duplicate tx_id: 0


,tx_id,src_id,dst_id,time_step,tx_type,amount,log_amount,src_balance_before,dst_balance_before,amount_to_src_balance_ratio,amount_exceeds_src_balance,src_balance_zero,label_fraud
0,tx_1,C1446786192,C629162506,207,CASH_OUT,201544.43,12.213770,0.0,4427398.04,0.0,1,1,0
1,tx_2,C1507137317,C95898629,301,CASH_OUT,99796.32,11.510897,0.0,2244500.63,0.0,1,1,0
2,tx_3,C689120377,C1777573992,429,CASH_OUT,31901.99,10.370455,0.0,9080459.95,0.0,1,1,0
3,tx_4,C1224308913,C766410750,187,CASH_OUT,286477.31,12.565418,0.0,949477.95,0.0,1,1,0
4,tx_5,C2098350559,C85657936,159,CASH_OUT,119075.98,11.687525,0.0,187137.64,0.0,1,1,0


### Transaction feature note

Sender-balance-derived features are retained at this stage for analysis.

Later modeling ablation will test whether these features capture transferable
fraud behavior or PaySim-specific synthetic patterns.

In [20]:
transaction_features_v1.to_csv(
    DATA_DIR / "transaction_features_v1.csv",
    index=False
)

print("Saved: ../data/transaction_features_v1.csv")

Saved: ../data/transaction_features_v1.csv


## 7. Historical Graph Features

For each transaction, historical graph features are calculated using only
transactions that occurred earlier in time.

The current transaction is never included in its own historical features.

Chronology is determined by `time_step`, not by `tx_id`.

In [21]:
with driver.session(database=DATABASE) as session:
    result = session.run("""
    MATCH (t:Transaction)-[:TO]->(dst:Account)

    OPTIONAL MATCH (past_t:Transaction)-[:TO]->(dst)
    WHERE past_t.time_step < t.time_step

    RETURN
        t.tx_id AS tx_id,
        count(DISTINCT past_t) AS receiver_prior_tx_count
    """)

    receiver_history_df = pd.DataFrame(
        [dict(record) for record in result]
    )

print("Receiver history:", receiver_history_df.shape)

receiver_history_df.head()

Receiver history: (200000, 2)


,tx_id,receiver_prior_tx_count
0,tx_1,0
1,tx_2,0
2,tx_3,0
3,tx_4,1
4,tx_5,0


In [22]:
transaction_features_graph = transaction_features_v1.merge(
    receiver_history_df,
    on="tx_id",
    how="left"
)

print(transaction_features_graph.shape)

(200000, 14)


In [23]:
transaction_features_graph.groupby(
    "label_fraud"
)["receiver_prior_tx_count"].agg([
    "count",
    "mean",
    "median",
    "max"
])

,count,mean,median,max
label_fraud,,,,
0,191787,0.369561,0.0,9
1,8213,0.123706,0.0,5


In [24]:
with driver.session(database=DATABASE) as session:
    result = session.run("""
    MATCH (src:Account)-[:SENT]->(t:Transaction)

    OPTIONAL MATCH (src)-[:SENT]->(past_t:Transaction)
    WHERE past_t.time_step < t.time_step

    RETURN
        t.tx_id AS tx_id,
        count(DISTINCT past_t) AS sender_prior_tx_count
    """)

    sender_history_df = pd.DataFrame(
        [dict(record) for record in result]
    )

print(sender_history_df.shape)

sender_history_df.head()

(200000, 2)


,tx_id,sender_prior_tx_count
0,tx_1,0
1,tx_2,0
2,tx_3,0
3,tx_4,0
4,tx_5,0


In [25]:
sender_history_analysis = transaction_features_v1[
    ["tx_id", "label_fraud"]
].merge(
    sender_history_df,
    on="tx_id",
    how="left"
)

sender_history_analysis.groupby(
    "label_fraud"
)["sender_prior_tx_count"].agg([
    "count",
    "mean",
    "median",
    "max"
])

,count,mean,median,max
label_fraud,,,,
0,191787,0.000047,0.0,1
1,8213,0.000000,0.0,0


**Decision:** `sender_prior_tx_count` is excluded from the final graph feature set.

Sender history is almost entirely absent in the sampled graph, so the feature
does not provide meaningful historical information.

### 7.1 Receiver Historical Amount Features

In addition to transaction counts, we measure the total and average amount
previously received by each destination account.

Only transactions with an earlier `time_step` are included.

In [26]:
with driver.session(database=DATABASE) as session:
    result = session.run("""
    MATCH (t:Transaction)-[:TO]->(dst:Account)

    OPTIONAL MATCH (past_t:Transaction)-[:TO]->(dst)
    WHERE past_t.time_step < t.time_step

    RETURN
        t.tx_id AS tx_id,
        coalesce(
            sum(past_t.amount),
            0
        ) AS receiver_prior_total_amount
    """)

    receiver_prior_amount_df = pd.DataFrame(
        [dict(record) for record in result]
    )

print(
    "Receiver prior amount:",
    receiver_prior_amount_df.shape
)

receiver_prior_amount_df.head()

Receiver prior amount: (200000, 2)


,tx_id,receiver_prior_total_amount
0,tx_1,0.00
1,tx_2,0.00
2,tx_3,0.00
3,tx_4,19989.23
4,tx_5,0.00


In [27]:
transaction_features_graph = (
    transaction_features_graph
    .merge(
        receiver_prior_amount_df,
        on="tx_id",
        how="left"
    )
)

print(transaction_features_graph.shape)

(200000, 15)


In [30]:
transaction_features_graph[
    "receiver_prior_avg_amount"
] = (
    transaction_features_graph[
        "receiver_prior_total_amount"
    ]
    /
    transaction_features_graph[
        "receiver_prior_tx_count"
    ].replace(0, np.nan)
)

transaction_features_graph[
    "receiver_prior_avg_amount"
] = (
    transaction_features_graph[
        "receiver_prior_avg_amount"
    ]
    .fillna(0)
)

In [31]:
transaction_features_graph[[
    "receiver_prior_tx_count",
    "receiver_prior_total_amount",
    "receiver_prior_avg_amount"
]].head()

,receiver_prior_tx_count,receiver_prior_total_amount,receiver_prior_avg_amount
0,0,0.00,0.00
1,0,0.00,0.00
2,0,0.00,0.00
3,1,19989.23,19989.23
4,0,0.00,0.00


In [32]:
transaction_features_graph.groupby(
    "label_fraud"
)[[
    "receiver_prior_total_amount",
    "receiver_prior_avg_amount"
]].agg([
    "mean",
    "median",
    "max"
])

receiver_prior_total_amount                      \
                                   mean median          max   
label_fraud                                                   
0                         124140.097586    0.0  56101683.57   
1                          43640.827778    0.0  24063093.15   

            receiver_prior_avg_amount                       
                                 mean median           max  
label_fraud                                                 
0                        88145.780354    0.0  3.584871e+07  
1                        31449.974663    0.0  1.203155e+07

In [33]:
transaction_features_graph_v1 = (
    transaction_features_graph[[
        "tx_id",
        "src_id",
        "dst_id",
        "time_step",
        "tx_type",

        "amount",
        "log_amount",
        "src_balance_before",
        "dst_balance_before",
        "amount_to_src_balance_ratio",
        "amount_exceeds_src_balance",
        "src_balance_zero",

        "receiver_prior_tx_count",
        "receiver_prior_total_amount",
        "receiver_prior_avg_amount",

        "label_fraud"
    ]]
    .copy()
)

print("Shape:", transaction_features_graph_v1.shape)
print(
    "Missing:",
    transaction_features_graph_v1.isna().sum().sum()
)
print(
    "Duplicate tx_id:",
    transaction_features_graph_v1[
        "tx_id"
    ].duplicated().sum()
)

Shape: (200000, 16)
Missing: 0
Duplicate tx_id: 0


In [34]:
transaction_features_graph_v1.to_csv(
    DATA_DIR / "transaction_features_graph_v1.csv",
    index=False
)

print(
    "Saved: ../data/transaction_features_graph_v1.csv"
)

Saved: ../data/transaction_features_graph_v1.csv


## 8. Temporal Receiver Activity

Historical receiver activity is also measured inside fixed time windows.

In PaySim, one `time_step` represents one hour. Therefore:

- 24 time steps = 24 hours
- 168 time steps = 7 days

These windows are PaySim-specific and should be replaced by real timestamps
in a reusable production pipeline.

In [35]:
with driver.session(database=DATABASE) as session:
    result = session.run("""
    MATCH (t:Transaction)-[:TO]->(dst:Account)

    OPTIONAL MATCH (past_t:Transaction)-[:TO]->(dst)

    WHERE past_t.time_step < t.time_step
      AND past_t.time_step >= t.time_step - 24

    RETURN
        t.tx_id AS tx_id,
        count(DISTINCT past_t)
            AS receiver_prior_24h_tx_count
    """)

    receiver_24h_df = pd.DataFrame(
        [dict(record) for record in result]
    )

print(receiver_24h_df.shape)

receiver_24h_df.head()

(200000, 2)


,tx_id,receiver_prior_24h_tx_count
0,tx_1,0
1,tx_2,0
2,tx_3,0
3,tx_4,0
4,tx_5,0


In [36]:
transaction_features_temporal = (
    transaction_features_graph_v1
    .merge(
        receiver_24h_df,
        on="tx_id",
        how="left"
    )
)

print(transaction_features_temporal.shape)

(200000, 17)


In [37]:
transaction_features_temporal.groupby(
    "label_fraud"
)["receiver_prior_24h_tx_count"].agg([
    "count",
    "mean",
    "median",
    "max"
])

,count,mean,median,max
label_fraud,,,,
0,191787,0.078791,0.0,6
1,8213,0.015829,0.0,3


**Observation:** The 24-hour receiver window is extremely sparse,
especially for fraudulent transactions. A wider temporal window is therefore
evaluated.

In [38]:
with driver.session(database=DATABASE) as session:
    result = session.run("""
    MATCH (t:Transaction)-[:TO]->(dst:Account)

    OPTIONAL MATCH (past_t:Transaction)-[:TO]->(dst)

    WHERE past_t.time_step < t.time_step
      AND past_t.time_step >= t.time_step - 168

    RETURN
        t.tx_id AS tx_id,
        count(DISTINCT past_t)
            AS receiver_prior_7d_tx_count
    """)

    receiver_7d_df = pd.DataFrame(
        [dict(record) for record in result]
    )

print(receiver_7d_df.shape)

receiver_7d_df.head()

(200000, 2)


,tx_id,receiver_prior_7d_tx_count
0,tx_1,0
1,tx_2,0
2,tx_3,0
3,tx_4,1
4,tx_5,0


In [39]:
transaction_features_temporal = (
    transaction_features_temporal
    .merge(
        receiver_7d_df,
        on="tx_id",
        how="left"
    )
)

print(transaction_features_temporal.shape)

(200000, 18)


In [40]:
transaction_features_temporal.groupby(
    "label_fraud"
)["receiver_prior_7d_tx_count"].agg([
    "count",
    "mean",
    "median",
    "max"
])

,count,mean,median,max
label_fraud,,,,
0,191787,0.262082,0.0,8
1,8213,0.064167,0.0,4


In [41]:
pd.crosstab(
    transaction_features_temporal[
        "label_fraud"
    ],
    transaction_features_temporal[
        "receiver_prior_7d_tx_count"
    ] == 0,
    normalize="index"
) * 100

receiver_prior_7d_tx_count,False,True
label_fraud,,
0,20.870028,79.129972
1,5.406064,94.593936


In [42]:
with driver.session(database=DATABASE) as session:
    result = session.run("""
    MATCH (t:Transaction)-[:TO]->(dst:Account)

    OPTIONAL MATCH (past_t:Transaction)-[:TO]->(dst)

    WHERE past_t.time_step < t.time_step
      AND past_t.time_step >= t.time_step - 168

    RETURN
        t.tx_id AS tx_id,
        coalesce(
            sum(past_t.amount),
            0
        ) AS receiver_prior_7d_total_amount
    """)

    receiver_7d_amount_df = pd.DataFrame(
        [dict(record) for record in result]
    )

print(receiver_7d_amount_df.shape)

receiver_7d_amount_df.head()

(200000, 2)


,tx_id,receiver_prior_7d_total_amount
0,tx_1,0.00
1,tx_2,0.00
2,tx_3,0.00
3,tx_4,19989.23
4,tx_5,0.00


In [43]:
transaction_features_temporal = (
    transaction_features_temporal
    .merge(
        receiver_7d_amount_df,
        on="tx_id",
        how="left"
    )
)

print(transaction_features_temporal.shape)

(200000, 19)


In [44]:
transaction_features_temporal.groupby(
    "label_fraud"
)["receiver_prior_7d_total_amount"].agg([
    "mean",
    "median",
    "max"
])

,mean,median,max
label_fraud,,,
0,90118.417754,0.0,56101683.57
1,23562.367590,0.0,23988003.29


### Temporal Feature Selection

The 24-hour receiver feature was retained for exploratory comparison but was
too sparse to include in the primary modeling feature set.

The final temporal graph features are:

- `receiver_prior_7d_tx_count`
- `receiver_prior_7d_total_amount`

In [45]:
print(
    "Final temporal dataset shape:",
    transaction_features_temporal.shape
)

print(
    "Missing values:",
    transaction_features_temporal
    .isna()
    .sum()
    .sum()
)

print(
    "Duplicate tx_id:",
    transaction_features_temporal[
        "tx_id"
    ].duplicated().sum()
)

Final temporal dataset shape: (200000, 19)
Missing values: 0
Duplicate tx_id: 0


In [46]:
transaction_features_temporal.to_csv(
    DATA_DIR / "transaction_features_temporal_v1.csv",
    index=False
)

print(
    "Saved: ../data/transaction_features_temporal_v1.csv"
)

Saved: ../data/transaction_features_temporal_v1.csv


In [47]:
driver.close()

print("Neo4j driver closed.")

Neo4j driver closed.
